# `tealtools.experimental` — full showcase

Two layers of analyses built on `tealtools.control_tree`, plus a
new third layer that lifts everything into a structured functional
IR:

| Layer | Modules | What it gives you |
|---|---|---|
| 1. Fold framework | `tree_fold` | Generic recursive fold over the control tree |
| 2. Direct analyses | `stack_depth`, `itxn_count`, `loop_summary`, `recursion`, `auth_dominance`, `call_graph` | Per-line + per-region results on the tree |
| 3. Structured IR | `funcir` | Lifted IR with If/IfElse/Switch/Loop/Sub + Let/Assign |

This notebook walks all three layers end-to-end on **xgov** and
**folks-finance**.

In [1]:
import os, sys, pickle, time, shutil, subprocess
from pathlib import Path

# ---- Repo-root auto-detection ----
def _find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for d in [here, *here.parents]:
        if (d / "tealtools").is_dir() and (d / "tests").is_dir():
            return d
    nb_dir = Path(globals().get("__vsc_ipynb_file__", __file__ if "__file__" in globals() else ".")).parent
    for d in [nb_dir, *nb_dir.parents]:
        if (d / "tealtools").is_dir():
            return d
    raise RuntimeError("could not locate the repo root (tealtools/ + tests/)")

ROOT = _find_repo_root()
sys.path.insert(0, str(ROOT))

# Codeql binary location — override via $CODEQL.
os.environ.setdefault("CODEQL", str(Path.home() / "tools" / "codeql" / "codeql"))
HAS_GRAPHVIZ = shutil.which("dot") is not None

def render_svg(dot_source):
    if not HAS_GRAPHVIZ: return None
    import tempfile
    with tempfile.NamedTemporaryFile(suffix=".dot", delete=False, mode="w") as f:
        f.write(dot_source); path = f.name
    return subprocess.check_output(["dot", "-Tsvg", path]).decode()

from tealtools.ssa import SSAProgram
from tealtools.experimental import (
    stack_depth, itxn_count, recursion, auth_dominance,
    loop_summary, call_graph,
)
from tealtools.experimental.tree_fold import TreeFold
from tealtools.experimental.funcir import lift as funcir_lift, pretty as funcir_pretty
from tealtools.experimental.funcir import auth as funcir_auth


# ---- Fixture loading — pickle-first, codeql second ----
# Running the notebook from Windows-side Jupyter against the WSL
# repo would otherwise crash on subprocess.run(codeql, ...) because
# the binary lives inside WSL. Pre-pickling SSAProgram in WSL with
#   python -c "import pickle; from tealtools.ssa import SSAProgram;
#              p = SSAProgram('tests/dbs/xgov-db');
#              p.propagate_constants();
#              pickle.dump(p, open('/tmp/xgov_ssa.pkl','wb'))"
# lets the notebook skip the codeql step entirely.

def _load_pickle(name: str):
    """Try several conventional locations for an SSAProgram pickle."""
    candidates = [
        Path("/tmp") / f"{name}_ssa.pkl",
        ROOT / "tests" / "dbs" / f"{name}_ssa.pkl",
        Path.home() / f"{name}_ssa.pkl",
    ]
    for p in candidates:
        try:
            if p.exists():
                with open(p, "rb") as f:
                    return pickle.load(f), p
        except Exception as e:
            print(f"  pickle load failed at {p}: {type(e).__name__}: {e}")
    return None, None

def _load_or_pickle(name: str, db_subpath: str):
    """Load from pickle if available; else load via codeql (which
    works only when codeql is reachable from this Python process)
    and report the failure mode clearly."""
    p, src = _load_pickle(name)
    if p is not None:
        print(f"  {name} (pickled): {len(p.blocks)} BBs  [{src}]")
        return p
    try:
        db = ROOT / "tests" / "dbs" / db_subpath
        p = SSAProgram(str(db))
        p.propagate_constants()
        print(f"  {name} (via codeql): {len(p.blocks)} BBs  [{db}]")
        return p
    except Exception as e:
        print(f"  {name}: load failed — {type(e).__name__}: {e}")
        print(f"      (pre-pickle inside WSL to skip codeql — see the comment above)")
        return None

print("Loading fixtures…")
xgov = _load_or_pickle("xgov", "xgov-db")
folks = _load_or_pickle("folks_finance", "folks-finance-db")

if xgov is None:
    raise RuntimeError("xgov fixture missing — see error above; rest of the notebook needs it")


Loading fixtures…


  xgov (pickled): 161 BBs  [/tmp/xgov_ssa.pkl]


  folks_finance (pickled): 464 BBs  [/tmp/folks_finance_ssa.pkl]


## Layer 2 — direct analyses (one-liners)

Stack depth, itxn count, loop summary, recursion, auth dominance,
call graph. Each is ~50–150 lines on top of the fold framework.

In [2]:
sd = stack_depth.analyze(xgov)
ic = itxn_count.analyze(xgov)
print(f"xgov stack max: {max(d['max'] for d in sd.values())}, itxn max: {max(ic.values())}")
print(f"xgov recursion: {recursion.render(xgov)}")
print()
print("xgov loop summary:")
print(loop_summary.render(xgov))


xgov stack max: 19, itxn max: 256
xgov recursion: (no recursive subroutines)

xgov loop summary:
Loop report:
  approval.teal:L242  body=119, submits/iter=0, max_iters=    23  (budget, 27 BBs, reducible=True)
  approval.teal:L275  body=2, submits/iter=0, max_iters=  1400  (budget, 1 BBs, reducible=True)
  approval.teal:L498  body=33, submits/iter=0, max_iters=    84  (budget, 6 BBs, reducible=True)
  approval.teal:L775  body=16, submits/iter=1, max_iters=   256  (tie, 2 BBs, reducible=True)
  approval.teal:L974  body=1963, submits/iter=0, max_iters=     1  (budget, 15 BBs, reducible=False)
  approval.teal:L1126  body=106, submits/iter=0, max_iters=    26  (budget, 16 BBs, reducible=True)


## Layer 2b — auth dominance, three iterations

| Variant | What it checks |
|---|---|
| `detect` (V1) | Any ancestor `If`/`IfElse`/`Switch`/`Guard` with `assert` in its cond region |
| `detect_with_predicates` (V2) | Path predicate at the BB must constrain an auth-relevant SSA value |
| `detect_with_predicates_interprocedural` (V3) | V2 + caller-context propagation through subroutines |

V3 is currently the tightest signal.

In [3]:
for label, prog in [("xgov", xgov), ("folks-finance", folks)]:
    if prog is None: continue
    print(f"--- {label} ---")
    v1 = auth_dominance.detect(prog)
    v2 = auth_dominance.detect_with_predicates(prog)
    v3 = auth_dominance.detect_with_predicates_interprocedural(prog)
    print(f"  V1 structural:        {len(v1):>4}")
    print(f"  V2 predicate-aware:   {len(v2):>4}")
    print(f"  V3 interprocedural:   {len(v3):>4}")


--- xgov ---


  V1 structural:          26
  V2 predicate-aware:      2
  V3 interprocedural:      2
--- folks-finance ---


  V1 structural:          37
  V2 predicate-aware:     21
  V3 interprocedural:     19


## Layer 3 — Functional IR

`funcir.lift(prog)` rewrites the entire program into a structured
AST. SSA values become `Let` bindings; phis (materialised by the
existing SSA layer) become `Assign` to mutable `mat_phi_k` vars
that get reassigned inside loops. Control flow is explicit
(`If`/`IfElse`/`Switch`/`Loop` with `Break`/`Guard`/`Call`/`Return`).

Polish that landed this round:

- Stack-manipulation ops (`frame_dig`, `dup`, ...) have their
  noisy stack-snapshot args suppressed; output names are filtered
  by use-analysis so dead carries get dropped automatically.
- The bnz/bz arm direction is normalised — `if cond:` always
  means "if cond, take the branch-target arm".
- `Switch` labels its arms with the source-label targets from the
  op's immediates.
- Irreducible (`Improper`) regions render as labelled BBs +
  explicit `Goto` / `IfGoto` — readable structured-goto pseudocode.

In [4]:
# A small fixture first — gives you the whole tree at a glance.
small = SSAProgram(str(ROOT / "tests" / "tealtools" / "cost" / "with_loop" / "db"))
print(funcir_pretty(funcir_lift(small)))


v5_1 = pushint 0
m1 := v5_1
loop:
  v7_1 = pushint 1
  v8_1 = (v7_1 + *m1)
  v9_1, v9_2 = dup
  m1 := v9_2
  v10_1 = txna ApplicationArgs 0
  v11_1 = btoi(v10_1)
  v12_1 = (v11_1 < v9_1)
  if not v12_1:
    break
return



In [5]:
# A real subroutine from xgov.
ir = funcir_lift(xgov)
target = next(s for n, s in ir.subs.items() if "L297" in n)
print(funcir_pretty(target))


sub sub_approval_teal_L297():
  proto 0 0
  itxn_begin
  v299_1 = pushint 6
  itxn_field TypeEnum(v299_1)
  v301_1 = pushbytes 0x0820020001311b221240001d361a0080044c6bea7212400001003119221231182213104488001123433119221240000100311822124423438a00003100320912442343
  itxn_field ApprovalProgram(v301_1)
  v303_1 = pushbytes 0x08810043
  itxn_field ClearStateProgram(v303_1)
  v305_1 = intc_0
  itxn_field Fee(v305_1)
  itxn_submit
  v308_1 = intc_0
  v309_1 = bytec_2
  v310_1, v310_2 = app_global_get_ex(v309_1, v308_1)
  store 23(v310_1)
  store 22(v310_2)
  v313_1 = load 23
  v314_1 = !(v313_1)
  assert v314_1
  v316_1 = bytec_2
  v317_1 = itxn CreatedApplicationID
  app_global_put(v317_1, v316_1)
  retsub


## Validation — funcir-based auth detector

Re-implement the auth dominance check as a walk over the IR
(no CFG, no SSA def-use chains, no path-predicate analysis). The
implementation drops from ~250 lines (V3) to ~150 lines, but…
the **precision drops** because we lose the path-predicate solver
that V3 leans on. The IR captures *syntactic* guards (If conds,
Asserts in same Block); V3 also captures *dynamic* path predicates
(bnz outcomes through dataflow, indirect dominator asserts).

The takeaway: structured IR makes some analyses dramatically
simpler — control-flow-shape ones especially — but it doesn't
replace a path-predicate solver. The IR + solver together would
likely be the best of both worlds. That's the next iteration.

In [6]:
for label, prog in [("xgov", xgov), ("folks-finance", folks)]:
    if prog is None: continue
    print(f"--- {label} ---")
    v3 = auth_dominance.detect_with_predicates_interprocedural(prog)
    f = funcir_auth.detect(prog)
    print(f"  control-tree V3 (with path predicates):  {len(v3):>4}")
    print(f"  funcir-based (syntactic guards only):    {len(f):>4}")


--- xgov ---
  control-tree V3 (with path predicates):     2
  funcir-based (syntactic guards only):      29
--- folks-finance ---


  control-tree V3 (with path predicates):    19
  funcir-based (syntactic guards only):      61


## Writing your own analysis

The `tree_fold.TreeFold[T]` framework remains the simplest path
for per-line metrics — subclass + override 3 methods:

In [7]:
class LogCountFold(TreeFold[int]):
    def __init__(self, prog):
        super().__init__(prog)
        self.per_line: dict[tuple[str, int], int] = {}
    def initial(self): return 0
    def merge(self, states): return max(states) if states else 0
    def visit_op(self, a, state, bb=None):
        new = state + (1 if a.op == "log" else 0)
        k = (a.location.file, a.location.line)
        if new > self.per_line.get(k, 0):
            self.per_line[k] = new
        return new
f = LogCountFold(xgov); f.run()
print(f"xgov max logs before any line: {max(f.per_line.values()) if f.per_line else 0}")


xgov max logs before any line: 1


## Recap

- The control tree gives us a clean substrate for analyses that
  ask "what dominates what" (auth-dominance, recursion, loop
  summaries, etc.).
- The funcir IR gives us a clean substrate for analyses that ask
  "what's the structure?" — pretty-printing, region diff, future
  formal-analysis backends.
- The two are complementary; future work would likely fold the
  path-predicate engine into the funcir walker, getting the best
  of both.

Next iterations on the funcir side:
- Per-function symbolic-execution prototype.
- Region diff between two programs (semantic version compare).
- Subroutine pre/post-condition templates.